In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch
import pandas as pd


e:\VS Code stuff\NLP Workspace\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
question = "What is the primary technical component or system mentioned?"

tickets = [
"I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.Could you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?",
"I am reporting a recurring issue with the Laser Printer when printing from MacBook Pros running macOS 15. Several team members have recently encountered this problem, which appears to be connected to the latest macOS 15 system updates.We believe the root cause might be a driver compatibility issue due to the updated operating systems or printer firmware. To troubleshoot, we have restarted the printers and MacBook devices, reinstalled the printer drivers, and verified configurations.",
"The investments dashboard is experiencing crashes, traced back to the MySQL 8.0 database. The potential reasons include data overflow or an analytics software error. I have already restarted Sophos Home and examined the MySQL 8.0 logs.",
"Dear Support Team,I am reaching out to request help concerning a problem with my Smart Home Center after a recent firmware upgrade. Since installing the update, the hub has been unable to synchronize multiple devices, which significantly hampers the functionality of my smart home setup.This issue started immediately after the update was completed. I have observed that several devices, such as smart lighting, thermostats, and security cameras, are unable to connect or stay synchronized with the hub due to connectivity problems.",
"Dear Customer Support Team, I am submitting a report concerning a critical outage in the platform services that is currently hindering device connectivity throughout operations. This issue has interrupted access to the SaaS platform, greatly affecting the functionality of barcode scanners, RAID controllers, and other vital daily tools. Initial diagnostics suggest that the root cause may be linked to failures in Kubernetes orchestration. We have attempted to mitigate the problem by restarting pods and redeploying microservices, but unfortunately, these measures have not resolved the issue. The problem persists."
]

model = "deepset/bert-large-uncased-whole-word-masking-squad2"

In [7]:
! pip install transformers==4.40.0 --force-reinstall

     ---------------------------------------- 0.0/137.6 kB ? eta -:--:--
     -- ------------------------------------- 10.2/137.6 kB ? eta -:--:--
     ----- ------------------------------- 20.5/137.6 kB 217.9 kB/s eta 0:00:01
     -------- ---------------------------- 30.7/137.6 kB 217.9 kB/s eta 0:00:01
     ------------- ----------------------- 51.2/137.6 kB 327.7 kB/s eta 0:00:01
     -------------------------- --------- 102.4/137.6 kB 490.2 kB/s eta 0:00:01
     ------------------------------------ 137.6/137.6 kB 541.8 kB/s eta 0:00:00
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 1.0 MB/s eta 0:00:00
  Using cached requests-2.3

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'e:\\vs code stuff\\nlp workspace\\venv\\lib\\site-packages\\81d243bd2c585b0f4821__mypyc.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Text Preprocessing & Transformer Usage

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model) # Load the tokenizer corresponding to the selected BERT model
qa_model = AutoModelForQuestionAnswering.from_pretrained(model)

results = []

for ticket in tickets:
    inputs = tokenizer(question, ticket, return_tensors="pt") # Tokenize the question and ticket together, returns pytorch tensors
    with torch.no_grad():
        outputs = qa_model(**inputs)
    
    start_idx = torch.argmax(outputs.start_logits)  # Find the token index with the highest start score
    end_idx = torch.argmax(outputs.end_logits) + 1 # Find the token index with the highest end score  [ +1 to include the end token ]
    
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx]  # Get the answer tokens from the input with start and end indices
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    
    results.append({"answer": answer, "score": outputs.start_logits.max().item()})
    
result_df = pd.DataFrame(results) # Convert results list to a pandas DataFrame for clean display


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3297.74it/s]
BertForQuestionAnswering LOAD REPORT from: deepset/bert-large-uncased-whole-word-masking-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Showing Results

In [52]:
result_df

,answer,score
0,centralized account management portal,5.780500
1,printer firmware,4.675098
2,data overflow or an analytics software error,4.127996
3,firmware upgrade,4.321458
4,"barcode scanners, raid controllers",4.221656


In [53]:
results

[{'answer': 'centralized account management portal',
  'score': 5.780500411987305},
 {'answer': 'printer firmware', 'score': 4.675098419189453},
 {'answer': 'data overflow or an analytics software error',
  'score': 4.127996444702148},
 {'answer': 'firmware upgrade', 'score': 4.321457862854004},
 {'answer': 'barcode scanners, raid controllers', 'score': 4.221656322479248}]